# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR\^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema URL and is structured according to the [MLCommons Croissant schema standard](https://mlcommons.org/croissant/).


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)} (Version: {getattr(metadata, 'version', None)})")
print(f"License: {getattr(metadata, 'license', None)}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s. This helps guide further data extraction and analysis.

In [ ]:
# List all record sets in the dataset
from mlcroissant._dataset.metadata_helpers import _find_by_type

record_sets = _find_by_type(metadata, 'RecordSet')
print(f"Total RecordSets: {len(record_sets)}\n")

for rs in record_sets:
    recset_id = getattr(rs, '@id', None)
    recset_name = getattr(rs, 'name', None)
    print(f"- RecordSet @id: {recset_id}")
    print(f"  Name      : {recset_name}")
    print(f"  Fields:")
    for f in getattr(rs, 'field', []):
        field_id = getattr(f, '@id', None)
        field_name = getattr(f, 'name', None)
        print(f"    - Field @id: {field_id} | name: {field_name}")
    print()

if not record_sets:
    print("No RecordSets found in the schema.")

Below is an **example** of how to iterate through records in a record set by its `@id`.

> Replace `'<RECORDSET_ID>'` below with the actual `@id` of the desired record set from the overview above.

In [ ]:
# Example: Preview records from a specific record set by @id
record_set_id = '<RECORDSET_ID>'  # Replace this with a real record set @id

try:
    for idx, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if idx > 2:
            break  # Only print first 3 records
except Exception as e:
    print(f"Error: Please replace <RECORDSET_ID> with an actual record set @id from the overview.\n{e}")

## 3. Data Extraction
Load all records from selected record sets into DataFrames for analysis.

Use the record set and field `@id`s as discovered above.

In [ ]:
# Define the record set(s) to extract data from; use the actual @id(s) from the overview.
record_sets_ids = [
    # Example: 'cr:MainTable',  # Replace this with available record set @id(s)
]

dataframes = {}
for rs_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for RecordSet '@id': {rs_id}")
        print(f"Columns: {df.columns.tolist()}\n")
    except Exception as exc:
        print(f"Could not load records for '@id': {rs_id}. Error: {exc}")

# For demonstration: preview the first dataframe if loaded
if dataframes:
    display_id = list(dataframes.keys())[0]
    print(f"Preview of first few records in: {display_id}")
    display(dataframes[display_id].head())
else:
    print("No dataframes loaded. Please fill in the list of record set @id(s).")

## 4. Exploratory Data Analysis (EDA)
Apply basic exploratory processing: filter, normalize, and group by key fields. 

Update the placeholders below with actual field `@id`s from your selected record set.

In [ ]:
# Select the record set for EDA
selected_rs_id = '<RECORDSET_ID>'  # e.g., the main table record set @id

# Select numeric and grouping field IDs
numeric_field = '<NUMERIC_FIELD_ID>'       # e.g. 'cr:abundance_field', pick a numeric one from field list
group_field = '<GROUP_FIELD_ID>'           # e.g. 'cr:protein_type', or similar categorical field

threshold = 10  # Example threshold

df = dataframes.get(selected_rs_id)
if df is not None and numeric_field in df.columns:
    # Filter records
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Optional: Group by a categorical field
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field} (showing means):")
        print(grouped_df.head())
    else:
        print(f"Column '{group_field}' not in dataframe columns. Grouping skipped.")
else:
    print("DataFrame not loaded or numeric_field/group_field ID is incorrect. Please update placeholders.")

## 5. Visualization
Plot distributions or relationships using selected fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: histogram and boxplot for a numeric field
if df is not None and numeric_field in df.columns:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f'Boxplot of {numeric_field}')

    plt.tight_layout()
    plt.show()

    # Boxplot by group field (if available)
    if group_field in df.columns and df[group_field].nunique() < 20:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('Cannot plot: DataFrame or chosen field(s) not available.')

## 6. Conclusion
- This notebook demonstrated how to load and explore a [Croissant-schema](https://mlcommons.org/croissant/) dataset using the `mlcroissant` library.
- By referencing record sets and fields via their `@id`, you can dynamically extract, analyze, and visualize FAIR data packages with minimal code changes.
- Update field and record set IDs as needed to suit your analysis needs!
